# Rank Examples by Global View Differences

This notebook loads the first 50 examples from the dataset, generates global views with and without scatter weights, calculates difference metrics, ranks them, and visualizes the top 5 examples with the largest differences.


## Setup


In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import Optional, Tuple
from tqdm import tqdm

# Adjust paths as needed
input_tce_csv_file = "/pdo/users/pablomer/mnt/tess/astronet/tces-vetting-v01-tois-triageJs-nocentroid-april2025-all.csv"
tess_data_dir = '/pdo/users/pablomer/mnt/tess/april2025_dataset_fits_files'

# Change to Astronet-Triage directory if needed
os.chdir('/pdo/users/pablomer/Astronet-Triage')
sys.path.insert(0, "/pdo/users/pablomer/Astronet-Triage")

from astronet.preprocess import preprocess


## Helper Functions


In [ ]:
def get_lightcurve(astro_id: int, tce_table: pd.DataFrame, tess_data_dir: str,
                   aperture: Optional[str] = None) -> tuple[np.ndarray, np.ndarray]:
    """Load light curve data for a given astro ID."""
    aperture_key_map = {
        "s": "SAP_FLUX_SML",
        "m": "SAP_FLUX_MID",
        "l": "SAP_FLUX_LAG",
        None: "SAP_FLUX",
    }

    matching_tces = tce_table[tce_table["Astro ID"] == astro_id]
    try:
        _, tce = next(matching_tces.iterrows())
    except StopIteration as e:
        raise ValueError(f"Astro ID not found: {astro_id}") from e

    if "MinT" not in tce:
        tce["MinT"] = -np.inf
    if "MaxT" not in tce:
        tce["MaxT"] = np.inf

    # Use the File column from the TCE table if available, otherwise construct filename
    if "File" in tce and pd.notna(tce["File"]):
        filename = tce["File"]
    else:
        # Fallback: construct filename (adjust format as needed for your data)
        astro_id_str = str(int(astro_id))[:-2].zfill(16)
        filename = (
        f"{tess_data_dir}/astronet_hlsp_qlp_tess_ffi-s0087-{astro_id_str}_tess_v01_llc.fits"
        )

    return preprocess.read_and_process_light_curve(
        tess_data_dir,
        aperture_key_map[aperture],
        filename,
        tce.MinT,
        tce.MaxT,
    )


def process_example(tce: pd.Series, tce_table: pd.DataFrame, tess_data_dir: str) -> Optional[dict]:
    """
    Process a single TCE example and generate both global views.
    Returns a dictionary with results or None if processing fails.
    """
    try:
        # Load light curve
        time, flux = get_lightcurve(tce['Astro ID'], tce_table, tess_data_dir)

        if len(time) == 0 or len(flux) == 0:
            return None

        # Detrend and filter
        detrended_time, detrended_flux, transit_mask = preprocess.detrend_and_filter(
            tce['TIC ID'], time, flux, tce.Per, tce.Epoc, tce.Dur, fixed_bkspace=None
        )

        if len(detrended_time) == 0:
            return None

        # Ensure epoch is within detrended time range
        epoch = tce.Epoc
        while epoch < detrended_time[0]:
            epoch += tce.Per

        # Calculate scatter weights
        scatter_weights = preprocess.split_and_calculate_weights(
            detrended_time, detrended_flux, gap_width=2
        )

        # Phase fold and sort
        folded_time, folded_flux, fold_num, tr_mask = preprocess.phase_fold_and_sort_light_curve(
            detrended_time, detrended_flux, transit_mask, tce.Per, epoch
        )

        # Align raw time for cadence selection
        raw_time_aligned, raw_flux_aligned = preprocess.align_raw_time(
            detrended_time, detrended_flux, tce.Per, epoch
        )

        # Align the weights
        weights_aligned = preprocess.align_scatter_weights(
            detrended_time, tce.Per, epoch, scatter_weights
        )

        # Generate global view WITHOUT scatter weights
        view_no_weights, std_no_weights, mask_no_weights, _, _ = preprocess.global_view(
            tce['TIC ID'],
            folded_time,
            folded_flux,
            tce.Per,
            all_30min=True,
            raw_time=raw_time_aligned,
            raw_flux=raw_flux_aligned,
            scatter_weights=None  # No scatter weights
        )

        # Generate global view WITH scatter weights
        view_with_weights, std_with_weights, mask_with_weights, _, _ = preprocess.global_view(
            tce['TIC ID'],
            folded_time,
            folded_flux,
            tce.Per,
            all_30min=True,
            raw_time=raw_time_aligned,
            raw_flux=raw_flux_aligned,
            scatter_weights=weights_aligned  # With scatter weights
        )

        return {
            'astro_id': tce['Astro ID'],
            'tic_id': tce['TIC ID'],
            'period': tce.Per,
            'epoch': tce.Epoc,
            'duration': tce.Dur,
            'view_no_weights': view_no_weights,
            'view_with_weights': view_with_weights,
            'folded_time': folded_time,
            'folded_flux': folded_flux,
        }

    except Exception as e:
        print(f"Error processing Astro ID {tce['Astro ID']}: {e}")
        return None


## Load First 50 Examples


In [ ]:
# Load TCE table
tce_table = pd.read_csv(input_tce_csv_file, header=0, low_memory=False)

# Filter to only examples with disp_p=1
if 'disp_p' in tce_table.columns:
    filtered_table = tce_table[tce_table['disp_p'] == 1]
    print(f"Filtered to {len(filtered_table)} examples with disp_p=1 (out of {len(tce_table)} total)")
else:
    print("Warning: 'disp_p' column not found in table. Proceeding without filter.")
    filtered_table = tce_table

# Get first 50 unique astro IDs from filtered table
unique_astro_ids = filtered_table["Astro ID"].unique()[:50]
print(f"Processing {len(unique_astro_ids)} examples...")

# Get TCE data for each astro ID (take first occurrence)
examples = []
for astro_id in unique_astro_ids:
    tce = filtered_table[filtered_table["Astro ID"] == astro_id].iloc[0]
    examples.append(tce)

print(f"Loaded {len(examples)} examples to process")


## Process All Examples and Calculate Differences


In [ ]:
# Process all examples
results = []
for tce in tqdm(examples, desc="Processing examples"):
    result = process_example(tce, tce_table, tess_data_dir)
    if result is not None:
        results.append(result)

print(f"\nSuccessfully processed {len(results)} out of {len(examples)} examples")


## Calculate Difference Metrics


In [ ]:
# Calculate difference metrics for each example
for result in results:
    view_no_weights = result['view_no_weights']
    view_with_weights = result['view_with_weights']

    # Calculate difference
    diff = view_with_weights - view_no_weights

    # Mean absolute difference (primary metric)
    result['mean_abs_diff'] = np.abs(diff).mean()

    # RMS difference (alternative metric - penalizes larger differences more)
    result['rms_diff'] = np.sqrt(np.mean(diff**2))

    # Max absolute difference
    result['max_abs_diff'] = np.abs(diff).max()

    # L2 norm of difference vector
    result['l2_norm'] = np.linalg.norm(diff)

    # Store the difference array for plotting
    result['difference'] = diff

print("Difference metrics calculated for all examples")
print(f"\nMetric statistics:")
print(f"  Mean absolute difference: {np.mean([r['mean_abs_diff'] for r in results]):.6f}")
print(f"  RMS difference: {np.mean([r['rms_diff'] for r in results]):.6f}")
print(f"  Max absolute difference: {np.mean([r['max_abs_diff'] for r in results]):.6f}")


## Rank Examples by Difference Metric

**Note:** We're using mean absolute difference as the primary metric. This metric:
- Is intuitive and easy to interpret
- Treats all differences equally (unlike RMS which penalizes larger differences)
- Is robust to outliers

Alternative metrics available:
- **RMS difference**: Penalizes larger differences more (good for detecting significant deviations)
- **Max absolute difference**: Captures worst-case deviation
- **L2 norm**: Similar to RMS, measures overall magnitude of difference vector


In [ ]:
# Rank by mean absolute difference (descending - largest differences first)
results_sorted = sorted(results, key=lambda x: x['mean_abs_diff'], reverse=True)

print("Top 10 examples by mean absolute difference:")
print("=" * 80)
for i, result in enumerate(results_sorted[:10], 1):
    print(f"{i:2d}. Astro ID: {result['astro_id']:6d}, TIC ID: {result['tic_id']:10d}, "
          f"Mean abs diff: {result['mean_abs_diff']:.6f}, "
          f"RMS diff: {result['rms_diff']:.6f}, "
          f"Max abs diff: {result['max_abs_diff']:.6f}")


## Plot Top 5 Examples


In [ ]:
# Plot top 5 examples
top_5 = results_sorted[:5]

fig, axes = plt.subplots(5, 4, figsize=(16, 20))

for row_idx, result in enumerate(top_5):
    view_no_weights = result['view_no_weights']
    view_with_weights = result['view_with_weights']
    diff = result['difference']
    x = np.arange(len(view_no_weights))

    # Row title with metrics
    row_title = (f"Rank {row_idx+1}: TIC {result['tic_id']} (Astro ID {result['astro_id']}) | "
                 f"Mean: {result['mean_abs_diff']:.6f}, RMS: {result['rms_diff']:.6f}, "
                 f"Max: {result['max_abs_diff']:.6f}")

    # Plot 1: Without scatter weights
    axes[row_idx, 0].plot(x, view_no_weights, '-', linewidth=1.5, alpha=0.9)
    axes[row_idx, 0].set_xlabel('Bin Index')
    axes[row_idx, 0].set_ylabel('Normalized Flux')
    axes[row_idx, 0].set_title('Without Scatter Weights', fontsize=10)
    axes[row_idx, 0].grid(True, alpha=0.3)
    axes[row_idx, 0].set_ylim(-1.1, 0.4)

    # Plot 2: With scatter weights
    axes[row_idx, 1].plot(x, view_with_weights, '-', linewidth=1.5, alpha=0.9, color='orange')
    axes[row_idx, 1].set_xlabel('Bin Index')
    axes[row_idx, 1].set_ylabel('Normalized Flux')
    axes[row_idx, 1].set_title('With Scatter Weights', fontsize=10)
    axes[row_idx, 1].grid(True, alpha=0.3)
    axes[row_idx, 1].set_ylim(-1.1, 0.4)

    # Plot 3: Overlay comparison
    axes[row_idx, 2].plot(x, view_no_weights, '-', linewidth=1.5, alpha=0.7, label='Without')
    axes[row_idx, 2].plot(x, view_with_weights, '--', linewidth=1.5, alpha=0.7, label='With', color='orange')
    axes[row_idx, 2].set_xlabel('Bin Index')
    axes[row_idx, 2].set_ylabel('Normalized Flux')
    axes[row_idx, 2].set_title('Overlay Comparison', fontsize=10)
    axes[row_idx, 2].legend()
    axes[row_idx, 2].grid(True, alpha=0.3)
    axes[row_idx, 2].set_ylim(-1.1, 0.4)

    # Plot 4: Difference
    axes[row_idx, 3].plot(x, diff, '-', linewidth=1.5, alpha=0.9, color='red')
    axes[row_idx, 3].axhline(0, color='black', linestyle='--', linewidth=0.8, alpha=0.5)
    axes[row_idx, 3].set_xlabel('Bin Index')
    axes[row_idx, 3].set_ylabel('Difference (With - Without)')
    axes[row_idx, 3].set_title('Difference', fontsize=10)
    axes[row_idx, 3].grid(True, alpha=0.3)

    # Add row title above the first subplot
    axes[row_idx, 0].text(0.5, 1.15, row_title, transform=axes[row_idx, 0].transAxes,
                          ha='center', va='bottom', fontsize=9, weight='bold',
                          bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.suptitle('Top 5 Examples by Mean Absolute Difference', fontsize=16, y=0.995)
plt.tight_layout(rect=[0, 0, 1, 0.98])
plt.show()


## Summary Statistics


In [ ]:
# Create a summary DataFrame
summary_data = {
    'Rank': range(1, len(results_sorted) + 1),
    'Astro ID': [r['astro_id'] for r in results_sorted],
    'TIC ID': [r['tic_id'] for r in results_sorted],
    'Period': [r['period'] for r in results_sorted],
    'Duration': [r['duration'] for r in results_sorted],
    'Mean Abs Diff': [r['mean_abs_diff'] for r in results_sorted],
    'RMS Diff': [r['rms_diff'] for r in results_sorted],
    'Max Abs Diff': [r['max_abs_diff'] for r in results_sorted],
    'L2 Norm': [r['l2_norm'] for r in results_sorted],
}

summary_df = pd.DataFrame(summary_data)

print("=" * 80)
print("SUMMARY STATISTICS")
print("=" * 80)
print(f"\nTotal examples processed: {len(results_sorted)}")
print(f"\nDifference metric statistics:")
print(f"  Mean absolute difference:")
print(f"    Mean: {summary_df['Mean Abs Diff'].mean():.6f}")
print(f"    Std:  {summary_df['Mean Abs Diff'].std():.6f}")
print(f"    Min:  {summary_df['Mean Abs Diff'].min():.6f}")
print(f"    Max:  {summary_df['Mean Abs Diff'].max():.6f}")
print(f"\n  RMS difference:")
print(f"    Mean: {summary_df['RMS Diff'].mean():.6f}")
print(f"    Std:  {summary_df['RMS Diff'].std():.6f}")
print(f"    Min:  {summary_df['RMS Diff'].min():.6f}")
print(f"    Max:  {summary_df['RMS Diff'].max():.6f}")

# Display top 10
print(f"\n{'='*80}")
print("Top 10 Examples:")
print("=" * 80)
print(summary_df.head(10).to_string(index=False))
